In [ ]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
from tqdm.autonotebook import tqdm
import os
from copy import deepcopy
from scipy.optimize import curve_fit
from scipy.stats import t

C:\Users\Administrator\AppData\Local\Temp\ipykernel_38276\1367502170.py:5: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
names = locals()

In [3]:
country_raster_1km_    = rio.open('../results/country-rasters/country-raster-eck4-1km.tif')
country_raster_50km_   = rio.open('../results/country-rasters/country-raster-eck4-50km.tif')
mining_suitability_1km = rio.open('../results/mining-suitability/global-mining-suitability-1km.tif').read(1)
country_raster_50km    = country_raster_50km_.read(1)
available_area_BAU_1km = rio.open('../results/mining-area/available-area-BAU-1km.tif').read(1)
available_area_CON_1km = rio.open('../results/mining-area/available-area-CON-1km.tif').read(1)

In [4]:
DF_1 = pd.read_excel(r'../data/20250129.xlsx', sheet_name='Sheet1')
df_q = DF_1[DF_1['type'] == 'quarry'].copy()
df1 = df_q[['m', 'top_area',  'total_area']].dropna()
x_data = df1['m'].values
y_data = df1['total_area'].values

In [5]:
def func_1(x, a, b):
    return a * x ** b

In [6]:
popt_1, pcov_1 = curve_fit(func_1, x_data, y_data) 
popt_2, pcov_2 = curve_fit(func_1, y_data, x_data) 

a_1, b_1 = popt_1
a_2, b_2 = popt_2

dof_1 = len(x_data) - len(popt_1)
dof_2 = len(y_data) - len(popt_2)

In [7]:
def predict_interval(X, popt, pcov, dof):
    a, b = popt
    cov_aa = pcov[0,0]
    cov_ab = pcov[0,1]
    cov_bb = pcov[1,1]

    mask_valid = (X > 0)

    X_safe = np.where(mask_valid, X, 1e-12)
    y_best = a * X_safe**b
    dy_da = X_safe**b
    dy_db = a * X_safe**b * np.log(X_safe)
    sigma_y2 = (
        cov_aa * dy_da**2 +
        2 * cov_ab * dy_da * dy_db +
        cov_bb * dy_db**2
    )

    sigma_y = np.sqrt(sigma_y2)

    tval = t.ppf(0.975, dof)

    lower = y_best - tval * sigma_y
    upper = y_best + tval * sigma_y

    lower = np.where(mask_valid, lower, 0)
    upper = np.where(mask_valid, upper, 0)

    return lower.astype(np.float32), upper.astype(np.float32)

In [8]:
def estimate_mining_area(
        scenario='BAU', level_1='M', level_2='M', level_3='M', res='50km'
):
    
    def _get_interal_index(i, j, src_res=res, dst_res='1km'):
        scale_dict = {'10km': 10, '50km': 50}
        scale = scale_dict[src_res]
        src_ref = rio.open(f'../results/country-rasters/country-raster-eck4-{src_res}.tif')
        dst_ref = rio.open(f'../results/country-rasters/country-raster-eck4-{dst_res}.tif')
        a = int(abs(dst_ref.transform[5] - src_ref.transform[5]) / 1e3) 
        b = int(abs(dst_ref.transform[2] - src_ref.transform[2]) / 1e3) 
        # print(a, b)
        row_min = i * scale - a
        row_max = (i + 1) * scale - a
        col_min = j * scale - b
        col_max = (j + 1) * scale - b

        return row_min, row_max, col_min, col_max
    
    if scenario in ['BAU', 'MIU', 'LE', 'RC', 'RE', 'SS', 'CCR', 'CE']:
        s_1 = 'BAU'
    else:
        s_1 = 'CON'

    avaiable_capacity_1km = rio.open(f'../results/uncertainty-analysis/extraction-capacity-{s_1}-1km-{level_2}{level_3}.tif').read(1)
    alocated_extraction = rio.open(f'../results/uncertainty-analysis/allocated-extraction-2023-2050-{scenario}-{res}-{level_1}{level_2}{level_3}.tif').read(1)
    country_raster_     = rio.open(f'../results/country-rasters/country-raster-eck4-{res}.tif')

    mining_area_expansion = np.zeros(country_raster_.shape, dtype=np.float32)
    i_set, j_set = np.where(alocated_extraction > 0)
    for i, j in tqdm(zip(i_set, j_set)):
        i_1, i_2, j_1, j_2 = _get_interal_index(i, j)
        suitability_slice = mining_suitability_1km[i_1:i_2, j_1:j_2] / 1e4
        available_capacity_slice = avaiable_capacity_1km[i_1:i_2, j_1:j_2]

        Q_total = alocated_extraction[i, j]
        score_slice = suitability_slice * available_capacity_slice

        score_flat = score_slice.flatten()
        avail_flat = available_capacity_slice.flatten()

        mask = avail_flat > 0
        score_flat = score_flat[mask]
        avail_flat = avail_flat[mask]

        if score_flat.size == 0:
            total_area = 0.0
        else:
            idx = np.argsort(score_flat)[::-1]
            score_flat = score_flat[idx]
            avail_flat = avail_flat[idx]
            q_max = avail_flat
            Q_remain = Q_total
            q_assigned = np.zeros_like(q_max)
            for m in range(len(q_max)):
                q_m = min(Q_remain, q_max[m])
                q_assigned[m] = q_m
                Q_remain -= q_m
                if Q_remain <= 0:
                    break
            
            if level_3 == 'M':
                area_in_each_sub_cand_cell = func_1(q_assigned, a_1, b_1) / 1e2   # unit: ha --> km²
            else:
                area_1km_lower, area_1km_upper = predict_interval(q_assigned, popt_1, pcov_1, dof_1)    
                area_in_each_sub_cand_cell = area_1km_lower / 1e2 if level_3 == 'L' else area_1km_upper / 1e2      # unit: ha --> km²

            total_area = np.sum(area_in_each_sub_cand_cell)

        mining_area_expansion[i, j] = total_area

    total_expansion = np.sum(mining_area_expansion)
    total_extraction = np.sum(alocated_extraction) / 1e3
    print(f'total expansion of {scenario}-{level_1}{level_2}{level_3} ({total_extraction:.1f} Gt): {total_expansion:.1f} km2')


    output_path = f'../results/uncertainty-analysis/mining-area-expansion-2050-{scenario}-{res}-{level_1}{level_2}{level_3}.tif'
    with rio.open(
        output_path, 
        'w', 
        driver='GTiff',
        height=country_raster_.shape[0],
        width=country_raster_.shape[1],
        count=1,
        dtype=mining_area_expansion.dtype,
        crs=country_raster_.crs,
        transform=country_raster_.transform, 
        compress='lzw',
    ) as dst:
        dst.write(mining_area_expansion, 1)

    return mining_area_expansion

In [ ]:
for scenario in ['BAU', 'MIU', 'LE', 'RC', 'RE', 'SS', 'CCR', 'CE']:
    for L_1 in ['L', 'M', 'H']:
        for L_2 in ['L', 'M', 'H']:
            for L_3 in ['L', 'M', 'H']:
                gg = estimate_mining_area(scenario=scenario, level_1=L_1, level_2=L_2, level_3=L_3)

In [ ]:
for scenario in ['CON', 'IAP']:
    for L_1 in ['L', 'M', 'H']:
        for L_2 in ['L', 'M', 'H']:
            for L_3 in ['L', 'M', 'H']:
                gg = estimate_mining_area(scenario=scenario, level_1=L_1, level_2=L_2, level_3=L_3)